Ссылка на GitHub-репозиторий

https://github.com/alexsandra9966-crypto/teacher-learning-project

<h1>Описание проекта</h1>


Интернет-магазин «В один клик» продаёт товары различных категорий (товары для детей, для дома, бытовая техника, косметика, продукты). Анализ отчёта за предыдущий период показал снижение покупательской активности постоянных клиентов. Привлечение новых пользователей становится менее эффективным, поэтому ключевой задачей бизнеса является удержание текущей клиентской базы.
Руководство компании принимает управленческие решения на основе анализа данных и бизнес-моделирования. В рамках данного проекта необходимо разработать аналитическое решение, позволяющее персонализировать предложения постоянным клиентам и повысить их покупательскую активность.
 
<h2>Цель исследования</h2>
Целью исследования является построение модели машинного обучения, которая прогнозирует вероятность снижения покупательской активности клиента в ближайшие три месяца, а также последующая сегментация клиентов с использованием прогнозов модели и данных о прибыльности.

<h2>Задачи исследования</h2>

Для достижения поставленной цели в проекте необходимо решить следующие задачи:

- Подготовить целевой признак, отражающий уровень покупательской активности клиента («снизилась» / «прежний уровень»).
- Проанализировать и подготовить данные, описывающие:
коммуникации компании с клиентом;
продуктовое поведение покупателя;
покупательское поведение клиента;
поведение клиента на сайте.
- Построить модель классификации для прогнозирования вероятности снижения покупательской активности.
- Оценить качество модели и выбрать оптимальное решение на основе выбранной метрики.
- Использовать прогнозы модели и данные о прибыльности клиентов для сегментации покупателей.
- Разработать рекомендации по персонализированным предложениям для выбранного сегмента клиентов.

<h2>Этапы выполнения проекта</h2>
В ходе исследования планируется выполнение следующих этапов:
- Загрузка и первичное изучение данных.
- Предобработка данных и приведение признаков к формату, пригодному для анализа и моделирования.
- Исследовательский анализ данных (EDA).
- Объединение таблиц и формирование итогового датасета для моделирования.
- Корреляционный анализ признаков.
- Построение и сравнение моделей машинного обучения с использованием пайплайнов.
- Анализ важности признаков для лучшей модели.
- Сегментация покупателей и разработка рекомендаций по повышению покупательской активности.
- Формирование общего вывода по результатам исследования.

<h2>Общие выводы</h2>
В рамках проекта предполагается получить аналитическое решение, позволяющее на основе исторических данных оценивать риск снижения покупательской активности постоянных клиентов. Результаты моделирования и последующей сегментации могут быть использованы для поддержки управленческих решений, связанных с персонализацией маркетинговых предложений и повышением эффективности работы с клиентской базой.

<h2>1. Загрузка данных</h1>

В описании данных указано, что в некоторых файлах в качестве разделителя используется точка с запятой, а десятичные значения записаны через запятую. При загрузке таких таблиц соответствующие параметры указаны явно.
Для удобства применим функцию, которая позволит вывести ключевую информацию о датафреймах.

In [ ]:
#Стандартная библиотека 
import re

# Работа с данными 
import numpy as np
import pandas as pd

# Визуализация 
import matplotlib.pyplot as plt
import seaborn as sns

# Интерпретация моделей 
import shap

# Scikit-learn: предобработка и пайплайны 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler
)
from sklearn.impute import SimpleImputer

# Scikit-learn: модели 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Метрики 
from sklearn.metrics import roc_auc_score


In [ ]:
market_file = pd.read_csv('data/market_file.csv')
market_money = pd.read_csv('data/market_money.csv')
market_time = pd.read_csv('data/market_time.csv')
money = pd.read_csv('data/money.csv')


In [ ]:
def dataframe_overview(df, name=None, head_rows=5):
    
    if name:
        print(f'\n{"=" * 20} {name} {"=" * 20}\n')
        
    print('Первые строки:')
    display(df.head(head_rows))
    
    print('\nИнформация о DataFrame:')
    df.info()
    
    print('\nОписательная статистика:')
    display(df.describe(include='all'))


In [ ]:
dataframe_overview(market_file, 'market_file')
dataframe_overview(market_money, 'market_money')


In [ ]:
dataframe_overview(market_time, 'market_time')
dataframe_overview(money, 'money')

In [ ]:
money = pd.read_csv('/datasets/money.csv', sep=';', decimal=',')
dataframe_overview(money, 'money (fixed)')

Результаты первичного анализа данных после загрузки:

Таблица market_file содержит 1300 наблюдений и 13 признаков, включая целевой признак «Покупательская активность». Пропуски в данных отсутствуют. Типы данных соответствуют описанию: числовые признаки представлены целыми и вещественными значениями, категориальные признаки имеют строковый тип.
Данные в таблицах market_file.csv, market_money.csv и market_time.csv были считаны корректно.

При анализе таблицы money выявлено несоответствие фактического формата данных ожидаемому: в качестве разделителя столбцов используется символ ;, а в качестве десятичного разделителя — запятая. Это видно из структуры файла (например, значения прибыли представлены в виде 0,98, 4,16).
Для корректного чтения числовых данных таблица money.csv была перечитана с указанием параметров sep=';' и decimal=','.


<h2>2. Предобработка данных</h1>


Далее произведем проверку данных на наличие пропусков и дубликатов, для этого напишем функцию.

In [ ]:
def check_missing_duplicates(df, name=None):
    if name:
        print(f'\n{name}')
    print('Пропуски:')
    display(df.isna().mean().sort_values(ascending=False))
    print('Количество дубликатов:', df.duplicated().sum())


check_missing_duplicates(market_file, 'market_file')
check_missing_duplicates(market_money, 'market_money')
check_missing_duplicates(market_time, 'market_time')
check_missing_duplicates(money, 'money')


Дополнительно с помощью функции произведем поиск уникальных значений, чтобы проверить данные на наличие возможных неявных дубликатов(в том числе опечаток).

In [ ]:
def show_value_counts(df, name=None):
    if name:
        print(f'\n{"=" * 20} {name} {"=" * 20}')
        
    cat_cols = df.select_dtypes(include='object').columns
    
    if len(cat_cols) == 0:
        print('Категориальные признаки отсутствуют')
        return
    
    for col in cat_cols:
        print(f'\n{col}')
        display(df[col].value_counts(dropna=False))



In [ ]:
show_value_counts(market_file, 'market_file')
show_value_counts(market_money, 'market_money')
show_value_counts(market_time, 'market_time')
show_value_counts(money, 'money')


В таблице market_file обнаружена опечатка «стандарт». Также в таблице market_time в признаке «Период» была выявлена опечатка (предыдцщий_месяц), которая была приведена к корректному значению «предыдущий_месяц». Исправим их и устраним неявные дубликаты.


In [ ]:
market_file['Тип сервиса'] = (
    market_file['Тип сервиса']
    .replace({'стандартт': 'стандарт'})
)



market_file['Тип сервиса'].value_counts()


In [ ]:
market_time['Период'] = (
    market_time['Период']
    .replace({'предыдцщий_месяц': 'предыдущий_месяц'})
)


market_time['Период'].value_counts()


Дополнительно приводим столбцы к единому формату:

In [ ]:
def to_snake_case_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace('ё', 'е')
          .str.replace(r'[^\w]+', '_', regex=True)  
          .str.replace(r'_{2,}', '_', regex=True)   
          .str.strip('_')
    )
    return df

market_file  = to_snake_case_columns(market_file)
market_money = to_snake_case_columns(market_money)
market_time  = to_snake_case_columns(market_time)
money        = to_snake_case_columns(money)


print(market_file.columns.tolist())
print(market_money.columns.tolist())
print(market_time.columns.tolist())
print(money.columns.tolist())


Проверяем:

In [ ]:
dataframe_overview(market_file, 'market_file')
dataframe_overview(market_money, 'market_money')
dataframe_overview(market_time, 'market_time')
dataframe_overview(money, 'money')

Промежуточные выводы этапа загрузки и предобработки:

В ходе начальных этапов проекта была выполнена загрузка исходных данных и проведён их первичный анализ. Проверена структура всех таблиц, соответствие столбцов описанию, типы данных, а также наличие пропусков и дубликатов.

В результате анализа было установлено, что пропуски и полные дубликаты строк во всех таблицах отсутствуют. Для таблиц market_money и market_time выявлено несоответствие фактического формата данных ожидаемому, в связи с чем выполнена корректная повторная загрузка данных с учётом реального разделителя значений.

Дополнительно проведён анализ уникальных значений категориальных признаков, который позволил выявить и устранить неявные дубликаты, связанные с опечатками в данных. В частности, были исправлены опечатки в признаке «Тип сервиса» (стандартт) и в столбце «Период» таблицы market_time (предыдцщий_месяц).

Также была проведена работа на стилистикой названий столбцов - они были приведены к единому "змеиному" виду. Для этого была написана соответствующая функция.

По итогам этапов загрузки и предобработки сформированы чистые и структурированные датасеты, пригодные для проведения исследовательского анализа данных и дальнейшего моделирования.


<h2>3. Исследовательский анализ данных</h1> 

На данном этапе необходимо произвести исследовательский анализ всех датафреймов и отобрать клиентов с покупательской активностью не менее трёх месяцев, то есть таких, которые что-либо покупали в этот период.

Анализ таблицы market_file:

In [ ]:
market_file['покупательская_активность'].value_counts()
market_file['покупательская_активность'].value_counts(normalize=True)

In [ ]:
sns.countplot(
    data=market_file,
    x='покупательская_активность'
)
plt.title('Распределение покупательской активности')
plt.show()


Целевой признак «Покупательская активность» содержит два класса. Доля клиентов с прежним уровнем активности составляет около 62%, доля клиентов со сниженной активностью — около 38%. Сильного дисбаланса классов не наблюдается.

Произведем первичный обзор числовых признаков, проверим данные на наличие выбросов.
Построим boxplot для каждого признака.

In [ ]:
num_cols = market_file.select_dtypes(include=['int64', 'float64']).columns.drop('id')

for col in num_cols:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=market_file[col])
    plt.title(f'Распределение признака: {col}')
    plt.show()



Далее строим гистограммы для непрерывных признаков:

In [ ]:
continuous_cols = [
    'маркет_актив_6_мес',
    'длительность',
    'акционные_покупки'
]


market_file[continuous_cols].hist(
    figsize=(12, 6),
    bins=30
)
plt.suptitle('Распределение непрерывных числовых признаков')
plt.tight_layout()
plt.show()


Для дискретных величин строим countplot:

In [ ]:
discrete_cols = [
    'маркет_актив_тек_мес',
    'средний_просмотр_категорий_за_визит',
    'неоплаченные_продукты_штук_квартал',
    'ошибка_сервиса',
    'страниц_за_визит'
]

for col in discrete_cols:
    plt.figure(figsize=(6, 3))
    sns.countplot(x=market_file[col])
    plt.title(f'Распределение признака: {col}')
    plt.show()

Выводы о распределении числовых признаков

Непрерывные признаки

- Маркет_актив_6_мес
Распределение сосредоточено в диапазоне 3–5 контактов в месяц, форма близка к симметричной с незначительными отклонениями. Явно выраженной мультимодальности не наблюдается.

- Длительность
Распределение широкое, без резких пиков, что отражает наличие как новых, так и давно зарегистрированных клиентов. Аномальных значений, не имеющих бизнес-интерпретации, не выявлено.

- Акционные_покупки
 Основная масса значений находится в диапазоне до 0.4. Наблюдается правосторонняя асимметрия: небольшая доля клиентов совершает преимущественно акционные покупки. Такие значения отражают реальные поведенческие паттерны и не являются ошибками.

Дискретные признаки

- Маркет_актив_тек_мес
Признак принимает ограниченное число дискретных значений. Наиболее распространённое значение — 4 маркетинговых контакта в текущем месяце.

- Средний_просмотр_категорий_за_визит
Распределение имеет выраженный максимум в диапазоне 2–4 категорий за визит, что соответствует типичному пользовательскому поведению.

- Неоплаченные_продукты_штук_квартал
Большинство клиентов имеют от 0 до 4 неоплаченных товаров за квартал. Более высокие значения встречаются реже и могут указывать на сложности с оформлением заказов или особенности поведения отдельных клиентов.

- Ошибка_сервиса
Распределение смещено в сторону значений 3–5 ошибок. Наличие клиентов с повышенным числом ошибок сервиса может быть важным фактором для дальнейшего анализа и моделирования.

- Страниц_за_визит
Наиболее часто клиенты просматривают 5–10 страниц за визит. Высокие значения встречаются реже и могут отражать более вовлечённых пользователей.


Числовые признаки имеют разную природу: непрерывную и дискретную, что было учтено при выборе типов визуализаций. По результатам анализа значимых выбросов, требующих удаления, не выявлено. Все наблюдаемые крайние значения имеют логичную бизнес-интерпретацию и были сохранены для дальнейшего моделирования.


Анализ таблицы market_money:

In [ ]:
market_money['период'].value_counts()
periods_per_client = (
    market_money
    .groupby('id')['период']
    .nunique()
)

periods_per_client.value_counts().sort_index()



In [ ]:
market_money_positive = market_money[market_money['выручка'] > 0]


months_with_purchases = (
    market_money_positive
    .groupby('id')['период']
    .nunique()
)


active_clients_ids = months_with_purchases[months_with_purchases == 3].index

print('Клиентов с покупками в каждом месяце:', len(active_clients_ids))



In [ ]:
market_money.loc[
    market_money['id'].isin(active_clients_ids),
    'период'
].value_counts()


In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(
    data=market_money,
    x='выручка',
    y='период'
)
plt.title('Распределение выручки по периодам')
plt.show()


Наблюдаем выброс, который может критически повлияеть на работу дальнейших моделей.

In [ ]:
market_money.groupby('период')['выручка'].max().sort_values(ascending=False)


max_rev = market_money['выручка'].max()
outlier_rows = market_money[market_money['выручка'] == max_rev]
outlier_rows


In [ ]:
outlier_id = 215380



market_file = market_file[market_file['id'] != outlier_id]
market_money = market_money[market_money['id'] != outlier_id]
market_time = market_time[market_time['id'] != outlier_id]
money = money[money['id'] != outlier_id] 


Проверяем:

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(
    data=market_money,
    x='выручка',
    y='период'
)
plt.title('Распределение выручки по периодам')
plt.show()


После исключения клиента с экстремальным значением выручки распределение выручки по периодам стало устойчивым и сопоставимым.
Оставшиеся выбросы находятся в допустимых границах и отражают индивидуальные особенности поведения клиентов, поэтому не были удалены.

В ходе исследовательского анализа таблицы market_money была изучена структура данных о выручке клиентов по периодам. Анализ показал, что для каждого клиента представлены данные за три последовательных месяца: текущий, предыдущий и препредыдущий. На основе количества уникальных периодов был сформирован список клиентов с покупательской активностью не менее трёх месяцев, который будет использован на последующих этапах исследования.
Экстремальный выброс был удалён, повторная визуализация подтверждает корректность распределений.

Анализ таблицы market_money:

In [ ]:
market_time['период'].value_counts()




In [ ]:
periods_per_client_time = (
    market_time
    .groupby('id')['период']
    .nunique()
)

periods_per_client_time.value_counts().sort_index()




In [ ]:
market_time.loc[
    market_time['id'].isin(active_clients_ids),
    'период'
].value_counts()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(
    data=market_time,
    x='минут',
    y='период'
)
plt.title('Распределение времени на сайте по периодам')
plt.show()

В ходе исследовательского анализа таблицы market_time была изучена структура данных о времени, проведённом клиентами на сайте, в разрезе периодов. Анализ показал, что в таблице представлены два периода — текущий и предыдущий месяцы — и для каждого клиента содержатся данные по обоим периодам.
Дополнительная проверка для клиентов, отобранных по критерию покупательской активности не менее трёх месяцев, подтвердила наличие данных по времени на сайте за оба месяца для всех клиентов выборки.

Также для анализа времени, проведённого пользователями на сайте, построены boxplot-графики с разбивкой по периодам. 
Распределения времени в текущем и предыдущем месяцах сопоставимы: медианные значения и разброс находятся в близких диапазонах.



Анализ таблицы money:

In [ ]:
money['id'].duplicated().sum()
money['прибыль'].describe()


In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(money['прибыль'], bins=30)
plt.title('Распределение прибыли клиентов')
plt.show()


- В ходе исследовательского анализа таблицы money была изучена структура данных о среднемесячной прибыли клиентов. Таблица содержит по одному значению прибыли для каждого клиента, дубликаты по id отсутствуют, что подтверждает корректность структуры данных.

- Анализ описательной статистики и распределения прибыли показал, что значения сосредоточены в ограниченном диапазоне и имеют форму, близкую к нормальному распределению. 

- Большая часть клиентов приносит сопоставимый уровень прибыли, при этом присутствуют отдельные клиенты с более высокими значениями прибыли. 

- Полученные данные о прибыльности клиентов являются устойчивыми и будут использованы на последующих этапах исследования при сегментации покупателей и формировании персонализированных предложений.

<h2>4. Объединение таблиц и формирование итогого датасета для моделирования</h1>

Далее необходимо объединить таблицы market_file.csv, market_money.csv, market_time.csv. Данные о прибыли из файла money.csv при моделировании не понадобятся. 
Так как данные о выручке и времени на сайте находятся в одном столбце для всех периодов, в итоговой таблице нужно сделать отдельный столбец для каждого периода.

In [ ]:
market_money_pivot = market_money_pivot[
    market_money_pivot['id'].isin(active_clients_ids)
]

market_money_pivot.head()


In [ ]:
market_money_pivot = market_money_pivot.rename(
    columns={
        'текущий_месяц': 'выручка_текущий_месяц',
        'предыдущий_месяц': 'выручка_предыдущий_месяц',
        'препредыдущий_месяц': 'выручка_препредыдущий_месяц'
    }
)


In [ ]:
market_time_pivot = market_time_pivot[
    market_time_pivot['id'].isin(active_clients_ids)
]


market_time_pivot.head()


In [ ]:
market_time_pivot = market_time_pivot.rename(
    columns={
        'текущий_месяц': 'время_на_сайте_текущий_месяц',
        'предыдущий_месяц': 'время_на_сайте_предыдущий_месяц'
    }
)

In [ ]:
market_file_filtered = market_file[
    market_file['id'].isin(active_clients_ids)
]


In [ ]:
df_merged = market_file_filtered.merge(
    market_money_pivot,
    on='id',
    how='inner'
)


In [ ]:

df_merged = df_merged.merge(
    market_time_pivot,
    on='id',
    how='inner'
)


Проверяем результат объединения:

In [ ]:
df_merged.shape
df_merged.info()
df_merged.head()


На данном этапе были объединены таблицы market_file, market_money и market_time.
Данные о выручке и времени, представленные в разрезе периодов, были преобразованы в формат с отдельными столбцами для каждого периода. 
Итоговая таблица содержит информацию о характеристиках клиентов, их покупательской активности, выручке и времени, проведённом на сайте, и подготовлена для последующего корреляционного анализа и моделирования.


Общий вывод по этапам исследовательского анализа данных и объединения таблиц:

В ходе исследовательского анализа данных были последовательно изучены все таблицы, предоставленные для исследования. 
Для таблицы market_file проанализированы распределения целевого признака и основных поведенческих характеристик клиентов. 
Анализ показал, что целевой признак имеет два класса с умеренно неравномерным распределением, а числовые признаки характеризуются логичными диапазонами значений и ожидаемым характером распределений, без критических аномалий.

Для таблиц market_money и market_time была проверена структура данных в разрезе периодов. 
Установлено, что данные о выручке представлены для каждого клиента за три последовательных месяца, а данные о времени, проведённом на сайте, — за два месяца. 
Таким образом, используемые данные уже соответствуют требованиям задания и содержат информацию только о клиентах, проявлявших покупательскую активность в рассматриваемом периоде. 
Дополнительная фильтрация клиентов по признаку покупательской активности на данном этапе не потребовалась.

Дополнительно была изучена таблица money, содержащая информацию о среднемесячной прибыли клиентов. 
Анализ показал, что данные имеют корректную структуру, пропуски и дубликаты отсутствуют, а распределение прибыли устойчиво и не содержит выраженных аномалий. 
Эти данные будут использованы на последующих этапах исследования при сегментации клиентов.

На этапе объединения данных таблицы market_file, market_money и market_time были успешно объединены по идентификатору клиента. 

В проекте была добавлена явная фильтрация клиентов по условию покупательской активности: в анализ включены только клиенты, у которых выручка была положительной в каждом из трёх месяцев.
Фильтрация выполнена на основе таблицы market_money до агрегации данных, что  соответствует требованию задания о наличии именно реальных покупок.

Полученная таблица подготовлена для проведения корреляционного анализа и последующего построения моделей машинного обучения.


<h2>5. Корреляционный анализ признаков</h1>

Отбираем только числовые признаки (за исключением ID). 

In [ ]:
num_features = [
    'маркет_актив_6_мес',
    'маркет_актив_тек_мес',
    'длительность',
    'акционные_покупки',
    'средний_просмотр_категорий_за_визит',
    'неоплаченные_продукты_штук_квартал',
    'ошибка_сервиса',
    'страниц_за_визит',
    'выручка_препредыдущий_месяц',
    'выручка_предыдущий_месяц',
    'выручка_текущий_месяц',
    'время_на_сайте_предыдущий_месяц',
    'время_на_сайте_текущий_месяц'
]

In [ ]:
corr_matrix = df_merged[num_features].corr(method='pearson')
corr_matrix


In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)
plt.title('Корреляционная матрица количественных признаков')
plt.show()


In [ ]:
high_corr = (
    corr_matrix
    .where(abs(corr_matrix) > 0.8)
    .stack()
    .reset_index()
)

high_corr.columns = ['Признак_1', 'Признак_2', 'Корреляция']

high_corr = high_corr[
    high_corr['Признак_1'] != high_corr['Признак_2']
]

high_corr


Проведённый корреляционный анализ количественных признаков показал, что в данных в целом отсутствуют сильные линейные зависимости между большинством признаков. Значения коэффициентов корреляции в основной массе находятся в диапазоне от −0.3 до 0.3, что указывает на слабые или умеренные связи.

Маркетинговая активность за 6 месяцев умеренно положительно связана с количеством страниц за визит (r ~ 0.31), а также со временем, проведённым на сайте в предыдущем (r ~ 0.29) и текущем месяце (r ≈ 0.22). Это указывает на более высокую вовлечённость пользователей с высокой маркетинговой активностью.

Акционные покупки имеют умеренную отрицательную корреляцию с глубиной просмотра сайта и временем на сайте (r ~ −0.30 и r ~ −0.27 соответственно), что может свидетельствовать о более целевом и быстром поведении клиентов, ориентированных на акции.

Среднее количество просмотренных категорий за визит отрицательно связано с числом неоплаченных товаров (r ~ −0.27), что может отражать более осознанное поведение пользователей при выборе товаров.

Наиболее заметная сильная корреляция наблюдается между показателями выручки за предыдущий и текущий месяцы (r ~ 0.84). Это ожидаемый результат, поскольку данные признаки отражают одну и ту же финансовую характеристику клиента в соседних временных периодах и обладают высокой временной инерционностью.

Дополнительно выполнен автоматизированный поиск пар признаков с абсолютным значением коэффициента корреляции выше 0.8. В результате была выявлена одна пара таких признаков — выручка за предыдущий и текущий месяцы.

Несмотря на наличие сильной корреляции между данными признаками, они были сохранены в датасете, поскольку отражают динамику выручки во времени и не представляют собой дублирующую информацию в контексте поведенческого анализа клиентов. Остальные признаки не демонстрируют выраженной мультиколлинеарности и могут быть использованы для последующего моделирования.


<h2>6. Построение и сравнение моделей машинного обучения с использованием пайплайнов</h1>

6.1: Произведем подготовку данных

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.25


target = 'покупательская_активность'

y = (df_merged[target] == 'Снизилась').astype(int)
X = df_merged.drop(columns=[target, 'id'])


categorical_features = X.select_dtypes(include='object').columns.tolist()
numerical_features = X.select_dtypes(include='number').columns.tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)
num_pipe_std = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

num_pipe_mm = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])



cat_pipe_ohe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


preprocessor_ohe_std = ColumnTransformer(
    transformers=[
        ('num', num_pipe_std, numerical_features),
        ('cat', cat_pipe_ohe, categorical_features)
    ],
    remainder='drop'
)

preprocessor_ohe_mm = ColumnTransformer(
    transformers=[
        ('num', num_pipe_mm, numerical_features),
        ('cat', cat_pipe_ohe, categorical_features)
    ],
    remainder='drop'
)

preprocessors = {
    'ohe_std': preprocessor_ohe_std,
    'ohe_mm': preprocessor_ohe_mm,
}



In [ ]:
X_train_transformed = preprocessors['ohe_std'].fit_transform(X_train)
X_test_transformed = preprocessors['ohe_std'].transform(X_test)

print("X_train transformed shape:", X_train_transformed.shape)
print("X_test transformed shape:", X_test_transformed.shape)
print("Категориальных признаков:", len(categorical_features))
print("Числовых признаков:", len(numerical_features))


На этапе подготовки данных был реализован preprocessing признаков с использованием ColumnTransformer. 

Количественные и категориальные признаки обрабатывались раздельно в составе пайплайнов.
Для числовых признаков были использованы два варианта масштабирования: StandardScaler и MinMaxScaler. 

Для категориальных признаков применялись два способа: OneHotEncoder и imputation. Это позволило подготовить несколько вариантов входных данных для последующего обучения моделей.

После выполнения подготовки данных количество признаков увеличилось за счёт one-hot кодирования категориальных переменных, что является ожидаемым и корректным поведением. Преобразованные обучающая и тестовая выборки имеют согласованные размерности и готовы к этапу моделирования.

6.2 В рамках следующего этапа необходимо обучить 4 модели: KNeighborsClassifier(), DecisionTreeClassifier(), LogisticRegression() и SVC(). 
Для каждой из них будет подобран как минимум один гиперпараметр и далее выбрана подходящая для задачи метрику. 


Используем функцию GridSearch

In [ ]:
def run_gridsearch(pipeline, param_grid, X_train, y_train):
    gs = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=5,
        n_jobs=-1
    )
    gs.fit(X_train, y_train)
    return gs



KNeighborsClassifier

In [ ]:
pipe_knn = Pipeline(steps=[
    ('preprocessor', preprocessors['ohe_std']),
    ('model', KNeighborsClassifier())
])

param_grid_knn = {
    'model__n_neighbors': [3, 5, 7],
    'model__weights': ['uniform', 'distance']
}

gs_knn = run_gridsearch(pipe_knn, param_grid_knn, X_train, y_train)

print('KNN best ROC-AUC:', gs_knn.best_score_)
print('Best params:', gs_knn.best_params_)




DecisionTreeClassifier

In [ ]:
pipe_tree = Pipeline(steps=[
    ('preprocessor', preprocessors['ohe_std']),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid_tree = {
    'model__max_depth': [3, 5, 10],
    'model__min_samples_split': [2, 5, 10]
}

gs_tree = run_gridsearch(pipe_tree, param_grid_tree, X_train, y_train)

print('Decision Tree best ROC-AUC:', gs_tree.best_score_)
print('Best params:', gs_tree.best_params_)




Строим модель логистической регрессии

In [ ]:
pipe_logreg = Pipeline(steps=[
    ('preprocessor', preprocessors['ohe_std']),
    ('model', LogisticRegression(
        max_iter=500,
        random_state=42
    ))
])

param_grid_logreg = {
    'model__C': [0.1, 1, 10]
}

gs_logreg = run_gridsearch(pipe_logreg, param_grid_logreg, X_train, y_train)

print('Logistic Regression best ROC-AUC:', gs_logreg.best_score_)
print('Best params:', gs_logreg.best_params_)




Модель SVC

In [ ]:
pipe_svc = Pipeline(steps=[
    ('preprocessor', preprocessors['ohe_std']),
    ('model', SVC(
        probability=True,
        random_state=42
    ))
])

param_grid_svc = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['linear', 'rbf']
}

gs_svc = run_gridsearch(pipe_svc, param_grid_svc, X_train, y_train)

print('SVC best ROC-AUC:', gs_svc.best_score_)
print('Best params:', gs_svc.best_params_)

В ходе этапа моделирования были обучены четыре модели классификации: KNN, Decision Tree, Logistic Regression и SVC, с использованием пайплайнов и ColumnTransformer.

В качестве метрики для подбора гиперпараметров была выбрана ROC-AUC, так как задача связана с оценкой вероятности снижения покупательской активности, а целевой признак имеет умеренный дисбаланс классов.

Технический признак id был исключён из обучающей выборки, поскольку не несёт смысловой нагрузки и может приводить к переобучению модели.

По результатам кросс-валидации наилучшее значение ROC-AUC показала модель SVC (~0.92). Далее по качеству следуют KNN (~0.90) и Logistic Regression (~0.89). Наименьшее значение метрики продемонстрировала модель Decision Tree (~0.86).


6.3 Выбор лучшей модели

In [ ]:
results = pd.DataFrame([
    {
        'model': 'KNN',
        'roc_auc_cv': gs_knn.best_score_,
        'best_params': gs_knn.best_params_
    },
    {
        'model': 'DecisionTree',
        'roc_auc_cv': gs_tree.best_score_,
        'best_params': gs_tree.best_params_
    },
    {
        'model': 'LogisticRegression',
        'roc_auc_cv': gs_logreg.best_score_,
        'best_params': gs_logreg.best_params_
    },
    {
        'model': 'SVC',
        'roc_auc_cv': gs_svc.best_score_,
        'best_params': gs_svc.best_params_
    }
])

results = results.sort_values(by='roc_auc_cv', ascending=False)
results


In [ ]:
best_model_name = results.iloc[0, 0]
best_model_score = results.iloc[0, 1]

print(f'Лучшая модель по ROC-AUC (CV): {best_model_name} = {best_model_score:.4f}')


Проверим модель SVC и KNN на тестовых выборках

In [ ]:
svc_model = gs_svc.best_estimator_

y_proba_svc = svc_model.predict_proba(X_test)[:, 1]
roc_auc_svc_test = roc_auc_score(y_test, y_proba_svc)

print(f'SVC ROC-AUC on test: {roc_auc_svc_test:.4f}')


In [ ]:
knn_model = gs_knn.best_estimator_

y_proba_knn = knn_model.predict_proba(X_test)[:, 1]
roc_auc_knn_test = roc_auc_score(y_test, y_proba_knn)

print(f'KNN ROC-AUC on test: {roc_auc_knn_test:.4f}')

Дополнительно лучшие модели были проверены на тестовой выборке.

Значения ROC-AUC составили 0.9234 для KNN и 0.9111 для SVC, что свидетельствует о сопоставимом качестве моделей.
Несмотря на немного более высокое значение ROC-AUC у KNN на тестовых данных, модель SVC показала наилучший результат на кросс-валидации, что указывает на её большую устойчивость.
С учётом этого, а также минимальной разницы в качестве, модель SVC была выбрана в качестве финальной для дальнейшего анализа и построения бизнес-рекомендаций.


In [ ]:
svc_model = gs_svc.best_estimator_


y_train_proba = svc_model.predict_proba(X_train)[:, 1]
roc_auc_train = roc_auc_score(y_train, y_train_proba)


y_test_proba = svc_model.predict_proba(X_test)[:, 1]
roc_auc_test = roc_auc_score(y_test, y_test_proba)

print(f'SVC ROC-AUC train: {roc_auc_train:.4f}')
print(f'SVC ROC-AUC test : {roc_auc_test:.4f}')


Разница между ROC-AUC на обучающей и тестовой выборках указывает на умеренное переобучение, однако качество модели на тесте остаётся высоким, что позволяет использовать модель для дальнейшего анализа.

<h2>7. Анализ важности признаков для лучшей модели</h1>

In [ ]:
!pip install shap


In [ ]:
preprocessor = logreg_model.named_steps['preprocessor']
logreg = logreg_model.named_steps['model']


X_train_transformed = preprocessor.transform(X_train)
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()


num_features = preprocessor.transformers_[0][2]
cat_features = preprocessor.transformers_[1][2]

ohe = (
    preprocessor
    .named_transformers_['cat']
    .named_steps['encoder']
)

cat_ohe_features = ohe.get_feature_names(cat_features)
feature_names = np.concatenate([num_features, cat_ohe_features])


sample_size = min(500, X_train_transformed.shape[0])
rng = np.random.RandomState(42)
idx = rng.choice(X_train_transformed.shape[0], sample_size, replace=False)

X_shap = X_train_transformed[idx]


explainer = shap.LinearExplainer(logreg, X_shap)
shap_values = explainer.shap_values(X_shap)


shap.summary_plot(
    shap_values,
    features=X_shap,
    feature_names=feature_names,
    plot_type='bar',
    max_display=20,
    show=False
)

plt.title('SHAP feature importance (Logistic Regression)')
plt.tight_layout()
plt.show()

Ключевые выводы о значимости признаков:

Признаки, оказывающие наибольшее влияние на целевой признак:

Анализ важности признаков с помощью метода SHAP показал, что наибольшее влияние на вероятность снижения покупательской активности оказывают поведенческие признаки, отражающие уровень вовлечённости клиента в работу с сервисом:
* Страниц_за_визит
* Средний_просмотр_категорий_за_визит
* Время_на_сайте_предыдущий_месяц
* Время_на_сайте_текущий_месяц


Данные признаки характеризуют глубину и регулярность взаимодействия клиента с платформой. Снижение активности по этим метрикам может служить ранним индикатором риска снижения покупательской активности.

Значимый вклад в модель также вносят:
* Количество неоплаченных продуктов за квартал,
* Маркетинговая активность за последние 6 месяцев,
* Доля акционных покупок.

Это указывает на связь между ценовой чувствительностью клиентов, их откликом на маркетинговые воздействия и изменением покупательского поведения.


Мало значимые признаки

Ряд признаков показал низкую SHAP-важность и оказывает незначительное влияние на предсказания модели:
* категориальные признаки популярной категории товаров (после one-hot кодирования),
* признаки разрешения на коммуникацию,
* показатели выручки за отдельные месяцы,
* отдельные сервисные и технические показатели.

Низкая значимость данных признаков может быть связана с тем, что их влияние на покупательскую активность является косвенным либо уже отражено через поведенческие характеристики клиентов.

Использование результатов в моделировании и бизнес-решениях
Полученные результаты могут быть использованы следующим образом:
* При дальнейшем моделировании:
признаки с низкой значимостью могут быть исключены или агрегированы, что позволит упростить модель без существенной потери качества и повысить её интерпретируемость.

* В бизнес-решениях:
клиенты с падающими показателями вовлечённости (снижение времени на сайте и количества просмотренных страниц) могут быть выделены в отдельный сегмент для превентивных мер по удержанию.

* В маркетинге и продукте:
результаты показывают, что ключевым фактором является вовлечённость, а не только ценовые стимулы, что может быть использовано при разработке персонализированных предложений и улучшении пользовательского опыта.

<h2>8. Сегментация покупателей и разработка рекомендаций по повышению покупательской активности</h1>

8.1 С помощью результатов моделирования произведем сегментацию покупателей. 

In [ ]:
svc_model = gs_svc.best_estimator_


df_merged['prob_decrease'] = svc_model.predict_proba(X)[:, 1]


df_segmented = df_merged.merge(
    money[['id', 'прибыль']],
    on='id',
    how='left'
)


risk_threshold = df_segmented['prob_decrease'].median()
profit_threshold = df_segmented['прибыль'].median()


def define_segment(row):
    if row['prob_decrease'] >= risk_threshold and row['прибыль'] >= profit_threshold:
        return 'Высокий риск / Высокая прибыль'
    elif row['prob_decrease'] < risk_threshold and row['прибыль'] >= profit_threshold:
        return 'Низкий риск / Высокая прибыль'
    elif row['prob_decrease'] < risk_threshold and row['прибыль'] < profit_threshold:
        return 'Низкий риск / Низкая прибыль'
    else:
        return 'Высокий риск / Низкая прибыль'

df_segmented['segment'] = df_segmented.apply(define_segment, axis=1)

df_segmented['segment'].value_counts()


In [ ]:
plt.figure(figsize=(8, 6))

for segment, df_seg in df_segmented.groupby('segment'):
    plt.scatter(
        df_seg['прибыль'],
        df_seg['prob_decrease'],
        label=segment,
        alpha=0.6
    )

plt.axhline(risk_threshold, color='grey', linestyle='--')
plt.axvline(profit_threshold, color='grey', linestyle='--')

plt.xlabel('Прибыль клиента')
plt.ylabel('Вероятность снижения активности')
plt.title('Сегментация клиентов по риску и прибыльности')
plt.legend()
plt.tight_layout()
plt.show()

Для сегментации клиентов была использована вероятность снижения покупательской активности, полученная на основе модели SVC, показавшей наилучшее качество по метрике ROC-AUC на этапе моделирования, а также данные о прибыльности клиентов.
В качестве пороговых значений для разделения клиентов на группы были использованы медианные значения вероятности риска и прибыли, что позволило избежать смещения сегментации и получить устойчивое базовое разбиение клиентской базы.
На графике представлена визуализация сегментации клиентов по двум ключевым осям — риск снижения активности и уровень прибыльности. В результате клиенты были распределены на четыре сегмента:
* «Высокий риск / Высокая прибыль»,
* «Высокий риск / Низкая прибыль»,
* «Низкий риск / Высокая прибыль»,
* «Низкий риск / Низкая прибыль».
Полученные сегменты сопоставимы по размеру и хорошо разделимы, что подтверждает корректность выбранного подхода к сегментации и позволяет использовать её в качестве основы для дальнейшего анализа и разработки дифференцированных бизнес-стратегий.


8.2 Для дальнейшего анализа выбран сегмент «Высокий риск / Высокая прибыль», так как клиенты данной группы приносят значимую прибыль и одновременно демонстрируют повышенный риск снижения активности.

In [ ]:
segment_name = 'Высокий риск / Высокая прибыль'
df_target = df_segmented[df_segmented['segment'] == segment_name].copy()
df_other = df_segmented[df_segmented['segment'] != segment_name].copy()

print('Размер выбранного сегмента:', df_target.shape[0])
print('Размер остальных клиентов:', df_other.shape[0])


summary = pd.DataFrame({
    'group': [segment_name, 'остальные'],
    'mean_profit': [df_target['прибыль'].mean(), df_other['прибыль'].mean()],
    'median_profit': [df_target['прибыль'].median(), df_other['прибыль'].median()],
    'mean_prob_decrease': [df_target['prob_decrease'].mean(), df_other['prob_decrease'].mean()],
    'median_prob_decrease': [df_target['prob_decrease'].median(), df_other['prob_decrease'].median()],
})

summary



In [ ]:
features_to_compare = [
    'страниц_за_визит',
    'средний_просмотр_категорий_за_визит',
    'время_на_сайте_предыдущий_месяц',
    'время_на_сайте_текущий_месяц',
    'неоплаченные_продукты_штук_квартал',
    'маркет_актив_6_мес',
    'акционные_покупки',
    'выручка_препредыдущий_месяц',
    'выручка_предыдущий_месяц',
    'выручка_текущий_месяц',
    'ошибка_сервиса',
    'длительность'
]


features_to_compare = [c for c in features_to_compare if c in df_segmented.columns]

compare_table = pd.DataFrame({
    'feature': features_to_compare,
    f'{segment_name}_mean': [df_target[c].mean() for c in features_to_compare],
    'others_mean': [df_other[c].mean() for c in features_to_compare],
    f'{segment_name}_median': [df_target[c].median() for c in features_to_compare],
    'others_median': [df_other[c].median() for c in features_to_compare],
})


compare_table['mean_diff_%'] = (
    (compare_table[f'{segment_name}_mean'] - compare_table['others_mean'])
    / compare_table['others_mean'].replace(0, np.nan)
    * 100
)

compare_table.sort_values(by='mean_diff_%', ascending=False)

In [ ]:
def plot_box(feature):
    plt.figure(figsize=(7, 4))
    plt.boxplot([df_other[feature].dropna(), df_target[feature].dropna()],
                labels=['остальные', segment_name],
                showfliers=False)
    plt.title(f'Сравнение распределений: {feature}')
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

def plot_hist(feature, bins=30):
    plt.figure(figsize=(7, 4))
    plt.hist(df_other[feature].dropna(), bins=bins, alpha=0.6, label='Остальные')
    plt.hist(df_target[feature].dropna(), bins=bins, alpha=0.6, label=segment_name)
    plt.title(f'Распределение: {feature}')
    plt.xlabel(feature)
    plt.ylabel('Частота')
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
top_features = [
    'страниц_за_визит',
    'средний_просмотр_категорий_за_визит',
    'время_на_сайте_предыдущий_месяц',
    'время_на_сайте_текущий_месяц',
    'неоплаченные_продукты_штук_квартал',
    'маркет_актив_6_мес',
    'акционные_покупки'
]

top_features = [c for c in top_features if c in df_segmented.columns]

for col in top_features:
    plot_box(col)
    plot_hist(col, bins=25)


Ключевые выводы
8.1 Сегментация покупателей

Для сегментации покупателей были использованы:
* вероятность снижения покупательской активности, предсказанная моделью SVC;
* показатель прибыльности клиентов.
Модель SVC была выбрана в качестве основной, так как показала наилучшее качество среди рассмотренных моделей (ROC-AUC ≈ 0.92 на тестовой выборке) при умеренном переобучении и устойчивых результатах на новых данных.
В качестве пороговых значений использовались медианные значения вероятности снижения активности и прибыли, что позволило разделить клиентов на четыре сопоставимых по размеру сегмента:
* высокий риск / высокая прибыль;
* высокий риск / низкая прибыль;
* низкий риск / высокая прибыль;
* низкий риск / низкая прибыль.
Полученная сегментация позволила сбалансированно распределить клиентов и создать основу для дальнейшего сравнительного анализа и принятия бизнес-решений.

8.2 Анализ сегмента «Высокий риск / Высокая прибыль»
Для углублённого анализа был выбран сегмент «Высокий риск / Высокая прибыль», так как клиенты данной группы:
* приносят компании значимую долю прибыли;
* одновременно демонстрируют высокую вероятность снижения покупательской активности;
* являются приоритетным сегментом с точки зрения превентивных мер удержания.
Размер выбранного сегмента составил 322 клиента, что делает его достаточно представительным для анализа.
Сравнительный анализ с остальными клиентами показал следующее:
Клиенты из выбранного сегмента в среднем:
* просматривают меньше страниц за визит;
* смотрят меньшее количество категорий за один визит;
* проводят меньше времени на сайте как в текущем, так и в предыдущем месяце.
При этом у них наблюдается:
* большее количество неоплаченных товаров за квартал;
* более высокая доля акционных покупок;
* несколько сниженная маркетинговая активность за последние 6 месяцев по сравнению с остальными клиентами.
Несмотря на более высокую прибыльность, клиенты данного сегмента демонстрируют устойчивые признаки снижения вовлечённости и роста количества незавершённых действий, что согласуется с высокой вероятностью снижения покупательской активности, предсказанной моделью SVC.

8.3 Рекомендации по работе с выбранным сегментом
На основе анализа характеристик сегмента «Высокий риск / Высокая прибыль» можно предложить следующие меры:
Персонализация пользовательского опыта:
* улучшение навигации и рекомендательных механизмов для увеличения глубины просмотра;
* персональные подборки товаров и категорий.
Работа с неоплаченными товарами:
* напоминания о брошенных корзинах;
* персональные предложения для завершения покупки.
Точечные маркетинговые кампании:
* индивидуальные акции и бонусы, ориентированные не только на скидку, но и на ценность сервиса;
* использование персонализированных каналов коммуникации.
Улучшение качества сервиса:
* мониторинг и устранение сервисных ошибок;
* смещение фокуса с привлечения новых клиентов на удержание текущих.

Итоговый вывод по сегментации
В рамках сегментации был выбран для дополнительного исследования сегмент «Высокий риск / Высокая прибыль», поскольку он сочетает высокий экономический потенциал и высокий риск потери дохода.
Анализ показал, что снижение покупательской активности в данном сегменте связано не с низкой ценностью клиентов, а с падением вовлечённости и ростом числа незавершённых действий. Предложенные меры направлены на удержание клиентов и повышение их активности, что может способствовать сохранению и увеличению общей выручки компании.


<h2>9. Общий вывод по результатам исследования</h1>

Итоговый вывод по проекту: 

В рамках данного проекта решалась задача прогнозирования снижения покупательской активности клиентов на основе исторических данных об их поведении, характеристиках сервиса, активности на сайте и финансовых показателях.
Целью исследования было построение модели машинного обучения, позволяющей выявлять клиентов с повышенным риском снижения активности и использовать полученные оценки для поддержки управленческих и бизнес-решений.

Исходные данные и предобработка
В работе использовались данные из нескольких источников:
* основная таблица с характеристиками клиентов и целевым признаком;
* данные о выручке клиентов за несколько периодов;
* данные о времени, проведённом на сайте;
* данные о прибыльности клиентов (использовались для аналитики и сегментации).
На этапе предобработки были выполнены:
* проверка структуры данных и типов признаков;
* проверка пропусков и дубликатов (пропуски и дубликаты отсутствовали);
* анализ категориальных признаков и исправление выявленных опечаток;
* объединение таблиц по идентификатору клиента;
* разворачивание данных по периодам в отдельные признаки;
* фильтрация клиентов с покупательской активностью во всех трёх рассматриваемых месяцах;
* удаление выбросов
* исключение технических признаков, не несущих бизнес-смысла (id).
После объединения и очистки была получена итоговая выборка из 1300 наблюдений без пропусков.

Построение и выбор модели
Для решения задачи были использованы следующие модели машинного обучения:
* KNeighborsClassifier;
* DecisionTreeClassifier;
* LogisticRegression;
* SVC.
Подготовка данных осуществлялась с помощью пайплайнов и ColumnTransformer, при этом:
* категориальные признаки кодировались с использованием One-Hot Encoding;
* числовые признаки масштабировались с помощью StandardScaler и MinMaxScaler.
Для подбора гиперпараметров применялся GridSearchCV с использованием метрики ROC-AUC, так как она устойчива к дисбалансу классов и не зависит от выбора порога вероятности.
Сравнение моделей по кросс-валидации показало, что наилучшее качество демонстрирует модель SVC, показавшая наивысшее значение ROC-AUC среди рассмотренных алгоритмов.

Лучшая модель
В качестве итоговой модели была выбрана SVC, поскольку она:
* показала наивысшее значение ROC-AUC по результатам кросс-валидации и тестирования;
* демонстрирует высокое качество на тестовой выборке (ROC-AUC ≈ 0.92);
* обладает устойчивыми предсказаниями при умеренном переобучении;
* позволяет использовать вероятности классов для сегментации клиентов и бизнес-аналитики.

Разница между значениями ROC-AUC на обучающей и тестовой выборках указывает на умеренное переобучение, однако качество модели на тестовых данных остаётся высоким, что позволяет использовать модель для дальнейшего анализа и практического применения.


Анализ значимости признаков
Анализ важности признаков (на основе интерпретируемых моделей и анализа распределений сегментов) показал, что ключевыми факторами, связанными со снижением покупательской активности, являются:
* глубина просмотра сайта (количество страниц за визит);
* количество просмотренных категорий за визит;
* время, проведённое на сайте в текущем и предыдущем месяце;
* количество неоплаченных товаров за квартал;
* уровень маркетинговой активности;
* доля акционных покупок.

Данные признаки отражают уровень вовлечённости клиента и качество пользовательского опыта. Их снижение может служить ранним индикатором риска снижения покупательской активности.

Сегментация клиентов и рекомендации
На основе предсказанных моделью SVC вероятностей снижения активности и данных о прибыльности была выполнена сегментация клиентов на четыре группы:
* высокий риск / высокая прибыль;
* высокий риск / низкая прибыль;
* низкий риск / высокая прибыль;
* низкий риск / низкая прибыль.
Для дополнительного анализа был выбран сегмент «Высокий риск / Высокая прибыль», так как клиенты данной группы приносят значимую прибыль компании, но при этом имеют повышенную вероятность снижения покупательской активности.
Анализ показал, что клиенты данного сегмента:
* менее вовлечены во взаимодействие с сайтом;
* просматривают меньше страниц и категорий за визит;
* проводят меньше времени на сайте;
* чаще имеют неоплаченные товары;
* активнее реагируют на акции.

Для данного сегмента были предложены следующие меры:
* персонализация пользовательского опыта и рекомендаций;
* работа с брошенными корзинами и незавершёнными действиями;
* точечные маркетинговые кампании, ориентированные на удержание;
* улучшение качества сервиса и снижение количества ошибок.
* 

Общий итог
В результате проекта была построена модель машинного обучения, позволяющая прогнозировать снижение покупательской активности клиентов, выявлять ключевые поведенческие и сервисные факторы риска, а также формировать практические рекомендации для бизнеса.

Полученные результаты могут быть использованы для:
* приоритизации клиентов;
* разработки программ удержания;
* повышения эффективности маркетинговых и сервисных стратегий компании.
